<a href="https://colab.research.google.com/github/eehujnihs21-stack/2555037/blob/main/25550337%EC%8B%A0%EC%A3%BC%ED%9D%AC_0529.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. 필수 패키지 설치
!pip install fastapi uvicorn sqlalchemy pydantic gradio python-multipart pandas

import os
import random
import pandas as pd
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session
import gradio as gr

# ==============================================================================
# [MVC 1단계: Database & Model] 데이터베이스 및 테이블 설정
# ==============================================================================
SQLALCHEMY_DATABASE_URL = "sqlite:///./library.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class DBBook(Base):
    __tablename__ = "books"
    id = Column(Integer, primary_key=True, index=True)
    title = Column(String, index=True, nullable=False)
    author = Column(String, nullable=False)
    description = Column(String, nullable=True)
    is_borrowed = Column(Boolean, default=False)

# 테이블 생성
Base.metadata.create_all(bind=engine)

# ==============================================================================
# [✨ 참신한 기능 추가: 초기 샘플 데이터 입력 로직 (Seeding) ✨]
# ==============================================================================
def seed_initial_books():
    db = SessionLocal()
    # 이미 데이터가 있다면 중복으로 넣지 않음
    if db.query(DBBook).count() == 0:
        sample_books = [
            # 1~5: IT 전공 및 교수님 헌정 도서
            DBBook(title="📘 파이썬 웹 프로그래밍의 정석", author="조상구", description="FastAPI와 파이썬을 이용한 백엔드 완벽 가이드북", is_borrowed=False),
            DBBook(title="📙 데이터베이스 개론 및 실습", author="조상구", description="관계형 데이터베이스 아키텍처와 효율적인 SQL 모델링 기법", is_borrowed=False),
            DBBook(title="📗 AI 시대의 소프트웨어 공학", author="이순신", description="인공지능 툴과 프롬프트 엔지니어링을 활용한 스마트 프로그래밍", is_borrowed=True),
            DBBook(title="📕 클린 코드 (Clean Code)", author="로버트 C. 마틴", description="애자일 소프트웨어 장인 정신 및 가독성 높은 코드 작성법", is_borrowed=False),
            DBBook(title="📔 도커와 쿠버네티스 실무 입문", author="강감찬", description="대규모 컨테이너 인프라 환경 구축과 오케스트레이션", is_borrowed=False),

            # 6~10: 유명 IT 및 개발 서적
            DBBook(title="📘 혼자 공부하는 자바", author="신용권", description="프로그래밍 초보자를 위한 자바 기초 및 객체 지향 개념", is_borrowed=False),
            DBBook(title="📙 리팩터링 2판", author="마틴 파울러", description="소프트웨어의 구조를 개선하여 가독성과 유지보수성을 높이는 기법", is_borrowed=True),
            DBBook(title="📗 디자인 패턴 (GoF)", author="에 Erich Gamma", description="객체지향 설계의 핵심 23가지 패턴 정복하기", is_borrowed=False),
            DBBook(title="📕 자바스크립트 딥 다이브", author="이웅모", description="자바스크립트의 기본 개념과 동작 원리 깊게 파고들기", is_borrowed=False),
            DBBook(title="📔 이것이 우분투 리눅스다", author="우재남", description="실무 환경 위주의 리눅스 서버 구축 및 핵심 명령어 가이드", is_borrowed=False),

            # 11~15: 인문, 과학, 경제 베스트셀러
            DBBook(title="📒 미움받을 용기", author="기시미 이치로", description="아들러 심리학을 기반으로 한 자유롭고 행복한 삶을 위한 조언", is_borrowed=False),
            DBBook(title="📕 사피엔스", author="유발 하라리", description="인류의 기원부터 인공지능 시대까지 문명의 거대한 역사", is_borrowed=False),
            DBBook(title="📗 코스모스 (Cosmos)", author="칼 세이건", description="우주의 신비와 인류의 탐험 역사를 다룬 과학 최고의 고전", is_borrowed=True),
            DBBook(title="📘 돈의 속성", author="김승호", description="최상위 부자가 말하는 돈에 대한 태도와 자산 관리 철학", is_borrowed=False),
            DBBook(title="📙 트렌드 코리아 2026", author="김난도", description="올해의 대한민국 소비 트렌드 전망과 핵심 키워드 분석", is_borrowed=False),

            # 16~20: 에세이 및 자기계발 명서
            DBBook(title="📔 불편한 편의점", author="김호연", description="청파동 골목 편의점에서 펼쳐지는 이웃들의 따뜻하고 유쾌한 이야기", is_borrowed=False),
            DBBook(title="📕 원씽 (The One Thing)", author="게리 켈러", description="복잡한 세상을 이기는 단 하나의 단순하고 강력한 성공 법칙", is_borrowed=False),
            DBBook(title="📘 타이탄의 도구들", author="팀 페리스", description="세계 최고들이 수집한 폭발적인 생산성과 성공의 비밀들", is_borrowed=False),
            DBBook(title="📗 데일 카네기 인간관계론", author="데일 카네기", description="사람을 사로잡고 소통을 원활하게 만드는 대인관계의 바이블", is_borrowed=True),
            DBBook(title="📙 역행자", author="자청", description="무의식과 본능을 깨부수고 경제적 자유를 얻는 7단계 공식", is_borrowed=False)
        ]
        db.add_all(sample_books)
        db.commit()
        print(f"💡 [시스템] 데이터베이스에 실제 도서 {len(sample_books)}권이 성공적으로 로드되었습니다.")
    db.close()

# 앱 시작 시 샘플 데이터 주입
seed_initial_books()


# ==============================================================================
# [MVC 2단계: CRUD 및 비즈니스 로직] 데이터베이스 제어 함수들
# ==============================================================================
def get_all_books(search_query=""):
    db = SessionLocal()
    query = db.query(DBBook)
    if search_query:
        query = query.filter(
            (DBBook.title.contains(search_query)) | (DBBook.author.contains(search_query))
        )
    books = query.all()
    db.close()
    return books

def add_new_book(title, author, description):
    if not title or not author:
        return "❌ 제목과 저자는 필수 입력 항목입니다."
    db = SessionLocal()
    db_book = DBBook(title=title, author=author, description=description)
    db.add(db_book)
    db.commit()
    db.close()
    return f"✅ 도서 '{title}' 등록 완료!"

def toggle_borrow(book_id):
    if book_id is None:
        return "❌ 도서 ID를 입력해주세요."
    db = SessionLocal()
    book = db.query(DBBook).filter(DBBook.id == book_id).first()
    if not book:
        db.close()
        return "❌ 해당 ID의 도서를 찾을 수 없습니다."

    book.is_borrowed = not book.is_borrowed
    status = "대출" if book.is_borrowed else "반납"
    db.commit()
    db.close()
    return f"✅ 도서 ID {book_id}번이 성공적으로 [{status}] 되었습니다."

def remove_book(book_id):
    if book_id is None:
        return "❌ 도서 ID를 입력해주세요."
    db = SessionLocal()
    book = db.query(DBBook).filter(DBBook.id == book_id).first()
    if not book:
        db.close()
        return "❌ 해당 ID의 도서를 찾을 수 없습니다."
    db.delete(book)
    db.commit()
    db.close()
    return f"✅ 도서 ID {book_id}번이 완전히 삭제되었습니다."

def get_random_recommendation():
    db = SessionLocal()
    available_books = db.query(DBBook).filter(DBBook.is_borrowed == False).all()
    db.close()

    if not available_books:
        return "📋 현재 추천 가능한 (대출 가능한) 도서가 없습니다."

    picked = random.choice(available_books)
    return f"📖 【 사서 추천 】 오늘의 한 책!\n\n📌 제목: {picked.title}\n✍️ 저자: {picked.author}\n💬 소개: {picked.description or '소개가 없습니다.'}"


# ==============================================================================
# [Gradio 인터페이스 데이터 가공] 표(Dataframe) 및 통계 업데이트
# ==============================================================================
def update_dashboard(search_query=""):
    books = get_all_books(search_query)

    # 1. 통계 데이터 계산
    total_count = len(get_all_books("")) # 전체 기준
    borrowed_count = len([b for b in get_all_books("") if b.is_borrowed])
    available_count = total_count - borrowed_count

    stats_html = f"""
    <div style="display: flex; gap: 15px; justify-content: space-around; margin-bottom: 10px;">
        <div style="background-color: #f0f4f8; padding: 15px; border-radius: 8px; text-align: center; flex: 1; border-left: 5px solid #2196F3;">
            <span style="font-size: 14px; color: #555;">📊 총 보유 도서</span><br>
            <strong style="font-size: 22px; color: #2196F3;">{total_count} 권</strong>
        </div>
        <div style="background-color: #e8f5e9; padding: 15px; border-radius: 8px; text-align: center; flex: 1; border-left: 5px solid #4CAF50;">
            <span style="font-size: 14px; color: #555;">🍏 대출 가능</span><br>
            <strong style="font-size: 22px; color: #4CAF50;">{available_count} 권</strong>
        </div>
        <div style="background-color: #ffebee; padding: 15px; border-radius: 8px; text-align: center; flex: 1; border-left: 5px solid #F44336;">
            <span style="font-size: 14px; color: #555;">🔴 대출 중</span><br>
            <strong style="font-size: 22px; color: #4CAF50;">{borrowed_count} 권</strong>
        </div>
    </div>
    """

    # 2. 표 데이터(Dataframe) 생성
    if not books:
        df = pd.DataFrame(columns=["ID", "도서 제목", "저자", "도서 설명", "대출 상태"])
        return stats_html, df

    data = []
    for b in books:
        status = "🔴 대출 중" if b.is_borrowed else "🍏 대출 가능"
        data.append([b.id, b.title, b.author, b.description or "-", status])

    df = pd.DataFrame(data, columns=["ID", "도서 제목", "저자", "도서 설명", "대출 상태"])
    return stats_html, df


# ==============================================================================
# [Gradio 인터페이스 구축] 신규 UI 대시보드 화면 Layout
# ==============================================================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📚 스마트 도서 관리 시스템 (FastAPI & Gradio)")
    gr.Markdown("실시간 DB 연동, 검색, 통계 및 초기 샘플 데이터 탑재 완료")

    # 상단 실시간 통계 보드 (HTML)
    init_stats, init_df = update_dashboard()
    stats_viewer = gr.HTML(value=init_stats)

    with gr.Row():
        # 왼쪽: 도서 현황 목록판 및 검색
        with gr.Column(scale=2):
            gr.Markdown("### 🗂️ 실시간 도서 보드")

            with gr.Row():
                in_search = gr.Textbox(
                    label="🔍 도서 검색 (제목 또는 저자)",
                    placeholder="검색어를 입력하고 엔터를 누르거나 새로고침하세요...",
                    scale=4
                )
                btn_refresh = gr.Button("🔄 새로고침 / 검색", variant="secondary", scale=1)

            # 테이블 형태로 표출
            book_table_viewer = gr.Dataframe(
                value=init_df,
                headers=["ID", "도서 제목", "저자", "도서 설명", "대출 상태"],
                datatype=["number", "string", "string", "string", "string"],
                interactive=False,
                wrap=True
            )

            # 추천 도서 컴포넌트
            gr.Markdown("---")
            with gr.Group():
                gr.Markdown("### 🎲 사서의 대출 추천")
                out_recommend = gr.Textbox(value=get_random_recommendation(), label="오늘 뭐 읽지?", lines=4)
                btn_recommend = gr.Button("🎲 다른 책 추천받기", variant="secondary")

        # 오른쪽: 도서 조작 제어반
        with gr.Column(scale=1):
            gr.Markdown("### 📥 신규 도서 등록")
            in_title = gr.Textbox(label="도서 제목", placeholder="예: 파이썬 웹 프로그래밍")
            in_author = gr.Textbox(label="저자 이름", placeholder="예: 홍길동")
            in_desc = gr.Textbox(label="도서 설명 (선택)", placeholder="간단한 책 소개")
            btn_add = gr.Button("➕ 도서 등록하기", variant="primary")
            out_add_result = gr.Markdown("")

            gr.Markdown("---")
            gr.Markdown("### ⚙️ 대출 / 반납 / 삭제 관리")
            in_book_id = gr.Number(label="도서 ID 번호 입력", value=1, precision=0)
            with gr.Row():
                btn_borrow = gr.Button("🔄 대출/반납", variant="warning")
                btn_delete = gr.Button("🗑️ 도서 삭제", variant="stop")
            out_manage_result = gr.Markdown("")

    # ==============================================================================
    # 기능 이벤트 매핑
    # ==============================================================================
    btn_refresh.click(fn=update_dashboard, inputs=[in_search], outputs=[stats_viewer, book_table_viewer])
    in_search.submit(fn=update_dashboard, inputs=[in_search], outputs=[stats_viewer, book_table_viewer])

    btn_add.click(
        fn=lambda t, a, d, s: (add_new_book(t, a, d), *update_dashboard(s)),
        inputs=[in_title, in_author, in_desc, in_search],
        outputs=[out_add_result, stats_viewer, book_table_viewer]
    )

    btn_borrow.click(
        fn=lambda bid, s: (toggle_borrow(bid), *update_dashboard(s)),
        inputs=[in_book_id, in_search],
        outputs=[out_manage_result, stats_viewer, book_table_viewer]
    )

    btn_delete.click(
        fn=lambda bid, s: (remove_book(bid), *update_dashboard(s)),
        inputs=[in_book_id, in_search],
        outputs=[out_manage_result, stats_viewer, book_table_viewer]
    )

    btn_recommend.click(fn=get_random_recommendation, inputs=[], outputs=[out_recommend])

# ==============================================================================
# [실행] 기존 프로세스 정리 및 가동
# ==============================================================================
!pkill -f uvicorn
!pkill -f gradio

print("\n🚀 데이터가 탑재된 스마트 도서 대시보드가 가동 중입니다...")
demo.launch(share=True, debug=False)

/tmp/ipykernel_1487/2711098122.py:19: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()
/tmp/ipykernel_1487/2711098122.py:186: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:



🚀 데이터가 탑재된 스마트 도서 대시보드가 가동 중입니다...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://21030a363b4fcd4873.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
